In [1]:
!pip install -q transformers accelerate tiktoken sentencepiece einops huggingface_hub pandas openpyxl tqdm
!pip install -q bitsandbytes Pillow qwen-vl-utils torchvision


SyntaxError: invalid syntax (2269011763.py, line 3)

In [ ]:
"""  Hata verme durumunda teminal içinde bu kodlar çalıştırılacaktır. """
pip install --upgrade transformers huggingface_hub accelerate tokenizers sentencepiece tiktoken
pip install git+[https://github.com/huggingface/transformers.git](https://github.com/huggingface/transformers.git)

In [ ]:

"""
AI PARALEL ÜRETİM SİSTEMİ - LOKAL HUGGING FACE ŞELALE MİMARİSİ v6.0 (NİHAİ)
Otonom Kuantizasyon, Multimodal Algılama, Disk/VRAM Temizliği ve Hata Mail Sistemi.
"""

import os
import sys
import json
import csv
import time
import gc
import shutil
import smtplib
from email.mime.text import MIMEText
import pandas as pd
import torch
from datetime import datetime, timezone
from tqdm import tqdm
from huggingface_hub import constants, login

from transformers import (
    AutoTokenizer, 
    AutoModelForCausalLM, 
    AutoProcessor, 
    AutoModelForVision2Seq,
    AutoModel,
    BitsAndBytesConfig
)

# =========================================================
# 1. KİMLİK DOĞRULAMA VE MAİL AYARLARI
# =========================================================

import os
from dotenv import load_dotenv

# .env dosyasındaki şifreleri sisteme yükle
load_dotenv(".env.txt")

# 🔑 HUGGING FACE TOKEN
HF_TOKEN = os.getenv("HF_TOKEN")

# 📧 EMAIL BİLDİRİM AYARLARI
# Madem iptal edeceksin, burayı doğrudan False yapabiliriz
ENABLE_EMAIL = False  

# Eğer ENABLE_EMAIL True olsaydı, verileri şu şekilde çekecekti:
if ENABLE_EMAIL:
    SENDER_EMAIL = os.getenv("SENDER_EMAIL")
    SENDER_PASSWORD = os.getenv("SENDER_PASSWORD")
    RECEIVER_EMAIL = os.getenv("RECEIVER_EMAIL")
else:
    SENDER_EMAIL = None
    SENDER_PASSWORD = None
    RECEIVER_EMAIL = None

# =========================================================
# 2. SABİT YOLLAR
# =========================================================

PROJECT_FOLDER = r'' # Kendi klasör yolunu buradan kontrol et
sys.path.append(PROJECT_FOLDER)
os.chdir(PROJECT_FOLDER)

INPUT_FILE = os.path.join(PROJECT_FOLDER, 'ell_balanced_1000.jsonl')
OUTPUT_FOLDER = os.path.join(PROJECT_FOLDER, 'Local_HF_System')
MODELS_EXCEL = os.path.join(PROJECT_FOLDER, 'models.xlsx')
ERROR_LOG_FILE = os.path.join(PROJECT_FOLDER, 'hata_alan_modeller.json')

os.makedirs(OUTPUT_FOLDER, exist_ok=True)

# Otomatik Hugging Face Login
# Otomatik Hugging Face Login
if HF_TOKEN:
    try:
        login(token=HF_TOKEN)
        print("🔐 Hugging Face hesabına başarıyla oturum açıldı.")
    except Exception as e:
        print(f"⚠️ HF Token ile giriş yapılırken uyarı: {e}")
else:
    print("ℹ️ HF_TOKEN bulunamadı (.env dosyası boş olabilir), açık kaynaklı modellerle devam ediliyor.")

# Prompt Modülü Yükleme
try:
    import essay_prompts_v1 as prompt_utils
    print("✅ 'essay_prompts_v1.py' başarıyla yüklendi.")
except ImportError:
    print("❌ 'essay_prompts_v1.py' bulunamadı.")
    sys.exit()

# =========================================================
# 3. YARDIMCI VE TEMİZLİK FONKSİYONLARI
# =========================================================

def send_error_email(model_id, error_message):
    """Kritik hataları yöneticiye mail atar."""
    if not ENABLE_EMAIL or SENDER_PASSWORD == "abcd efgh ijkl mnop":
        return
    
    subject = f"🚨 AI Fabrikası: {model_id} Modelinde Hata"
    body = f"Sistem '{model_id}' modelini çalıştırırken bir hata ile karşılaştı ve bu modeli atladı.\n\n" \
           f"Hata Detayı:\n{error_message}\n\n" \
           f"Sistem çökmüş değil. Şelale mimarisi gereği bir sonraki modelden üretime devam ediyor."
    
    msg = MIMEText(body, 'plain', 'utf-8')
    msg['Subject'] = subject
    msg['From'] = SENDER_EMAIL
    msg['To'] = RECEIVER_EMAIL

    try:
        server = smtplib.SMTP('smtp.gmail.com', 587)
        server.starttls()
        server.login(SENDER_EMAIL, SENDER_PASSWORD)
        server.send_message(msg)
        server.quit()
        print("📧 Hata maili başarıyla yöneticinize iletildi!")
    except Exception as e:
        print(f"⚠️ Hata maili gönderilemedi: {e}")

def load_target_models():
    """Excel dosyasından sadece model_id'leri okur."""
    if not os.path.exists(MODELS_EXCEL):
        print(f"❌ KONTROL HATASI: {MODELS_EXCEL} bulunamadı!")
        sys.exit()
    
    df = pd.read_excel(MODELS_EXCEL)
    models = []
    for _, row in df.iterrows():
        m_id = str(row.get("model_id", "")).strip()
        if m_id and m_id.lower() != "nan":
            models.append(m_id)
    return models

def log_error_model(model_id, reason):
    """Hatalı modelleri log dosyasına kaydeder ve mail atar."""
    error_entry = {
        "model_id": model_id,
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "reason": str(reason)
    }
    data = []
    if os.path.exists(ERROR_LOG_FILE):
        try:
            with open(ERROR_LOG_FILE, "r", encoding="utf-8") as f:
                data = json.load(f)
        except: data = []
    data.append(error_entry)
    with open(ERROR_LOG_FILE, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=4)
        
    # Loga yazdıktan sonra Mail tetikleyicisi
    send_error_email(model_id, reason)

def force_system_cleanup(model_id=None):
    """OOM hatasını ve Disk Dolmasını önlemek için agresif temizlik."""
    if model_id:
        print(f"\n🧹 [{model_id}] için sistem temizliği başlatılıyor...")
    else:
        print("\n🧹 Kuantizasyon arası VRAM temizliği başlatılıyor...")
        
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
    
    if model_id:
        repo_folder_name = f"models--{model_id.replace('/', '--')}"
        model_cache_path = os.path.join(constants.HF_HUB_CACHE, repo_folder_name)
        if os.path.exists(model_cache_path):
            try:
                shutil.rmtree(model_cache_path, ignore_errors=True)
                print(f"🗑️ Diskten başarıyla silindi: {model_cache_path}")
            except Exception as e:
                print(f"⚠️ Önbellek silinirken uyarı: {e}")

def setup_model_paths(model_id):
    safe_name = model_id.replace("/", "_").replace(":", "_").replace(" ", "_")
    model_output_folder = os.path.join(OUTPUT_FOLDER, safe_name)
    os.makedirs(model_output_folder, exist_ok=True)
    return {
        "jsonl": os.path.join(model_output_folder, f'{safe_name}-Generated.jsonl'),
        "csv": os.path.join(model_output_folder, f'{safe_name}-Generated_Flat.csv')
    }

def append_to_csv(family_data, csv_path):
    root = family_data.get("json_data", family_data)
    family_id = root.get("family_id", "UNKNOWN")
    versions = root.get("versions", [])
    orig = next((v for v in versions if v["type"] == "original"), None)
    if not orig: return

    row_data = {"family_id": family_id, "original_text": orig.get("text", "")}
    for v in versions:
        if v["type"] == "original": continue
        prompt_id = v.get("prompt_used_id", "unknown")
        row_data[f"{prompt_id}_text"] = v.get("text", "")
        row_data[f"{prompt_id}_model"] = v.get("llm_model", "")
        row_data[f"{prompt_id}_status"] = v.get("status", "")

    file_exists = os.path.isfile(csv_path)
    with open(csv_path, "a", newline="", encoding="utf-8") as csvfile:
        writer = csv.DictWriter(csvfile, fieldnames=list(row_data.keys()))
        if not file_exists:
            writer.writeheader()
        writer.writerow(row_data)

# =========================================================
# 4. YEREL MODEL İLE ÜRETİM MOTORU VE OTONOM KUANTİZASYON
# =========================================================

def load_model_with_auto_quant(model_id, model_class, needs_remote_code):
    """VRAM yetersizliğinde (OOM) otomatik olarak 8-bit ve 4-bit'e düşer."""
    levels = ["raw", "8bit", "4bit"]
    last_err = None
    
    for level in levels:
        try:
            if level == "raw":
                print("   -> [Seviye 1] Orijinal hassasiyette (BF16/FP16) yükleniyor...")
                return model_class.from_pretrained(
                    model_id,
                    torch_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
                    device_map="auto",
                    trust_remote_code=needs_remote_code,
                    token=HF_TOKEN
                )
            elif level == "8bit":
                print("   -> ⚠️ VRAM yetmedi! [Seviye 2] 8-Bit Kuantizasyon ile deneniyor...")
                force_system_cleanup() # VRAM'i boşalt
                return model_class.from_pretrained(
                    model_id,
                    device_map="auto",
                    quantization_config=BitsAndBytesConfig(load_in_8bit=True),
                    trust_remote_code=needs_remote_code,
                    token=HF_TOKEN
                )
            elif level == "4bit":
                print("   -> ⚠️ Yine yetmedi! [Seviye 3] 4-Bit Maksimum Sıkıştırma ile deneniyor...")
                force_system_cleanup()
                return model_class.from_pretrained(
                    model_id,
                    device_map="auto",
                    quantization_config=BitsAndBytesConfig(
                        load_in_4bit=True,
                        bnb_4bit_compute_dtype=torch.float16
                    ),
                    trust_remote_code=needs_remote_code,
                    token=HF_TOKEN
                )
        except Exception as e:
            err_str = str(e).lower()
            if "cuda out of memory" in err_str or "outofmemory" in err_str or "alloc" in err_str:
                last_err = e
                continue # Bir sonraki sıkıştırma seviyesine geç
            else:
                raise e # OOM dışı bir hataysa doğrudan fırlat
                
    raise RuntimeError(f"4-Bit modunda bile VRAM yetersiz geldi: {last_err}")

def generate_local_response(model, tokenizer_or_processor, messages, max_new_tokens=1024):
    inputs = tokenizer_or_processor.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors="pt"
    ).to(model.device)

    pad_id = getattr(tokenizer_or_processor, "pad_token_id", None)
    if pad_id is None and hasattr(tokenizer_or_processor, "tokenizer"):
        pad_id = getattr(tokenizer_or_processor.tokenizer, "pad_token_id", None)
    if pad_id is None:
        pad_id = getattr(tokenizer_or_processor, "eos_token_id", None)

    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=max_new_tokens, do_sample=True,
            temperature=0.7, top_p=0.9, pad_token_id=pad_id
        )
    
    decode_func = getattr(tokenizer_or_processor, "decode", None)
    if decode_func is None:
        decode_func = tokenizer_or_processor.tokenizer.decode
        
    return decode_func(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True).strip()

# =========================================================
# 5. ANA ÇALIŞTIRMA BLOĞU
# =========================================================

if __name__ == "__main__":
    print("=" * 65)
    print(" 🌟 LOKAL HUGGING FACE ÜRETİM FABRİKASI BAŞLATILIYOR 🌟 ")
    print("=" * 65)

    with open(INPUT_FILE, "r", encoding="utf-8") as f:
        master_families = [json.loads(line) for line in f]

    all_prompts = prompt_utils.list_prompts()
    all_prompt_ids = {p.id for p in all_prompts}
    
    models_to_run = load_target_models()
    print(f"📋 Excel'den toplam {len(models_to_run)} adet model okundu.\n")

    for model_id in models_to_run:
        paths = setup_model_paths(model_id)

        print("\n" + "=" * 65)
        print(f"🚀 MODELLER DÖNGÜSÜ: {model_id}")
        print("=" * 65)

        existing_families = []
        if os.path.exists(paths["jsonl"]):
            with open(paths["jsonl"], "r", encoding="utf-8") as f:
                for line in f:
                    try: existing_families.append(json.loads(line))
                    except: pass

        fully_processed_ids = set()
        for fam in existing_families:
            root = fam.get("json_data", fam)
            success_prompts = {v.get("prompt_used_id") for v in root.get("versions", []) if v.get("status") == "OK"}
            if success_prompts == all_prompt_ids:
                fully_processed_ids.add(root.get("family_id"))

        remaining_families = [
            f for f in master_families
            if (f.get("family_id") or f.get("json_data", {}).get("family_id")) not in fully_processed_ids
        ]

        if not remaining_families:
            print(f"✅ [{model_id}] için TÜM metinler (%100) zaten tamamlanmış! Sıradakine geçiliyor.")
            force_system_cleanup(model_id)
            continue

        print(f"🎯 Kalan Makale Ailesi: {len(remaining_families)} / {len(master_families)}")

        model = None
        tokenizer = None
        is_multimodal = False
        needs_remote_code = False
        target_model_class = AutoModelForCausalLM
        
        try:
            print(f"📥 Model Hugging Face'den analiz ediliyor: {model_id}")
            
            # Tokenizer / Processor Analizi
            try:
                tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=False, token=HF_TOKEN)
            except ValueError as e:
                if "trust_remote_code" in str(e):
                    print("⚠️ Tokenizer özel kod istiyor. 'trust_remote_code=True' aktif edildi.")
                    needs_remote_code = True
                    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True, token=HF_TOKEN)
                elif "Unrecognized configuration class" in str(e) or "Processor" in str(e):
                    is_multimodal = True
                else:
                    raise e
            except Exception as e:
                 if "Unrecognized" in str(e) or "Multimodal" in str(e):
                     is_multimodal = True
                 else:
                     raise e

            # Model Sınıfı Belirleme
            if is_multimodal:
                print("👁️ Multimodal (Görsel+Metin) mimarisi algılandı! AutoProcessor devreye giriyor...")
                try:
                    tokenizer = AutoProcessor.from_pretrained(model_id, trust_remote_code=needs_remote_code, token=HF_TOKEN)
                except ValueError as inner_e:
                    if "trust_remote_code" in str(inner_e):
                        needs_remote_code = True
                        tokenizer = AutoProcessor.from_pretrained(model_id, trust_remote_code=True, token=HF_TOKEN)
                    else:
                        raise inner_e
                target_model_class = AutoModelForVision2Seq
            
            # Modeli Otonom Kuantizasyon ile Yükle
            try:
                model = load_model_with_auto_quant(model_id, target_model_class, needs_remote_code)
            except ValueError as e:
                error_msg = str(e)
                if "Unrecognized configuration class" in error_msg:
                    print("⚠️ Spesifik model sınıfı reddedildi. Genel 'AutoModel' sınıfı ile deneniyor...")
                    model = load_model_with_auto_quant(model_id, AutoModel, needs_remote_code)
                elif "trust_remote_code" in error_msg:
                    needs_remote_code = True
                    model = load_model_with_auto_quant(model_id, target_model_class, True)
                else:
                    raise e
                    
            if getattr(tokenizer, "pad_token", None) is None:
                tokenizer.pad_token = getattr(tokenizer, "eos_token", None)
                
            print("✅ Model üretime hazır!")

        except Exception as e:
            print(f"❌ [CRITICAL HATA] Model yüklenemedi: {e}")
            log_error_model(model_id, f"Model Yükleme Hatası: {e}")
            force_system_cleanup(model_id)
            continue

        # --- 3. ÜRETİM DÖNGÜSÜ ---
        try:
            with open(paths["jsonl"], "a", encoding="utf-8") as f_out:
                for family in tqdm(remaining_families, desc=f"Üretiliyor ({model_id})"):
                    root = family.get("json_data", family)
                    fam_id = root.get("family_id")
                    orig = next((v for v in root.get("versions", []) if v["type"] == "original"), None)
                    if not orig: continue

                    completed_prompts = set()
                    for ex in existing_families:
                        ex_root = ex.get("json_data", ex)
                        if ex_root.get("family_id") == fam_id:
                            completed_prompts = {v.get("prompt_used_id") for v in ex_root.get("versions", []) if v.get("status") == "OK"}

                    new_family = root.copy()
                    new_family["versions"] = [orig]
                    at_least_one_new = False

                    for tmpl in all_prompts:
                        if tmpl.id in completed_prompts:
                            continue
                        
                        try:
                            messages = tmpl.build_chat_messages(essay_text=orig["text"])
                            ai_text = generate_local_response(model, tokenizer, messages)

                            if ai_text and len(ai_text) > 50:
                                new_family["versions"].append({
                                    "type": tmpl.id, "text": ai_text, "length_chars": len(ai_text),
                                    "llm_model": model_id, "prompt_used_id": tmpl.id,
                                    "improvement_aspects": tmpl.aspects,
                                    "generated_at": datetime.now(timezone.utc).isoformat(),
                                    "status": "OK"
                                })
                                at_least_one_new = True
                            else:
                                new_family["versions"].append({
                                    "type": tmpl.id, "text": "", "status": "FAILED", "failure_reason": "Kısa veya boş çıktı."
                                })
                        except Exception as req_err:
                            print(f"\n⚠️ İstek üretilirken hata oluştu: {req_err}")
                            new_family["versions"].append({
                                "type": tmpl.id, "text": "", "status": "FAILED", "failure_reason": str(req_err)
                            })

                    if at_least_one_new:
                        f_out.write(json.dumps(new_family, ensure_ascii=False) + "\n")
                        f_out.flush()
                        os.fsync(f_out.fileno())
                        append_to_csv(new_family, paths["csv"])

        except Exception as loop_err:
            print(f"🚨 Üretim sırasında beklenmeyen hata (OOM vb.): {loop_err}")
            log_error_model(model_id, f"Üretim Hatası: {loop_err}")

        # --- 4. BİTİŞ VE TEMİZLİK ---
        del model
        del tokenizer
        model = None
        tokenizer = None
        
        force_system_cleanup(model_id)

    print("\n🎉 TÜM MODEL DÖNGÜSÜ TAMAMLANDI!")

🔐 Hugging Face hesabına başarıyla oturum açıldı.
✅ 'essay_prompts_v1.py' başarıyla yüklendi.
 🌟 LOKAL HUGGING FACE ÜRETİM FABRİKASI BAŞLATILIYOR 🌟 
📋 Excel'den toplam 35 adet model okundu.


🚀 MODELLER DÖNGÜSÜ: meta-llama/Llama-3.2-3B
🎯 Kalan Makale Ailesi: 1000 / 1000
📥 Model Hugging Face'den analiz ediliyor: meta-llama/Llama-3.2-3B
   -> [Seviye 1] Orijinal hassasiyette (BF16/FP16) yükleniyor...


`torch_dtype` is deprecated! Use `dtype` instead!


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/1.46G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]